# SIH-SUS · Stratified mixture of experts

Adapted from *A Mixture-of-Experts Framework for Practical Hybrid-Quantum Models in Credit Card Fraud Detection* (arXiv:2603.06473v2), sections on preprocessing and validation. The unit is an admission, identified by `id_linha`. The question is generalization to held-out admissions drawn from the study population.

Repeated stratified 5-fold cross-validation (3 repeats) reserves 80% per fold for expert training. The remaining 20% is stratified again into 50% calibration, 25% router analysis and 25% final holdout. Within the expert pool, a stratified search split selects the expert hyperparameters. Majority undersampling to 50/50 is restricted to rows passed to expert fit or refit. Calibration, router analysis and final holdout retain the observed imbalanced distribution. Expert checkpoints are frozen for those three roles.

Calibration fits temperature and a Youden J threshold. The router learns where the hybrid expert alone predicts correctly on Analysis. A prespecified probability threshold of 0.60 selects the hybrid expert for hard routing. Soft routing is included as a comparison. Both experts are evaluated to obtain the standalone results, so this notebook measures predictive performance.

Raw date fields, month features and historical hospital aggregates are excluded. Each role is disjoint by admission. The frozen file supplies no patient identifier. The circuit comparison holds the 50-component balanced axis encoding fixed and compares the original chain with disconnected rings and pairs at 19, 26 and 33 qubits. Exact local expectations use exported TileLang CUDA kernels.


In [ ]:
import os
if os.environ.get('PQFM_SIH_FIRST_FOLD_HILL') == '1':
    os.environ.setdefault('OMP_NUM_THREADS', '8')

import hashlib
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, auc, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, mean_squared_error,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

DATA_DIR = Path(os.environ.get('PQFM_SIH_DATA', '/home/pichau/pqfmlib-api/data/sih-sus/protocolo_comparacao'))
SEED = 42
N_JOBS = int(os.environ.get('PQFM_XGB_THREADS', '4'))
GAMMAS = (0.05, 0.1, 0.6)
TARGETS = {
    "classificacao": "morte",
}

import joblib
from threadpoolctl import threadpool_limits


In [ ]:
from contextlib import contextmanager
from contextvars import ContextVar
from datetime import datetime
from functools import wraps
from inspect import signature

_ACTIVE_TIMING = ContextVar("demanda_active_timing", default=None)
TIMING_COLUMNS = ["task", "fold", "stage_id", "parent_id", "depth", "stage",
                  "started_at", "finished_at", "seconds", "exclusive_seconds", "status"]


def format_duration(seconds):
    """Formata duração monotônica em HH:MM:SS.sss; não representa horário do dia."""
    hours, rest = divmod(float(seconds), 3600)
    minutes, rest = divmod(rest, 60)
    return f"{int(hours):02d}:{int(minutes):02d}:{rest:06.3f}"


def format_log_timestamp(timestamp):
    """Exibe horário local com dois espaços entre data e hora, sem sufixo de fuso."""
    return datetime.fromisoformat(timestamp).strftime("%Y-%m-%d  %H:%M:%S")


@contextmanager
def measure_stage(label):
    """Registra início/fim e tempo inclusivo/exclusivo de uma etapa do fold ativo.

    Usa perf_counter para duração e horário local com fuso apenas para os logs.
    parent_id/depth descrevem a hierarquia: não somar pais e filhos. Sem fold
    ativo, não instrumenta chamadas isoladas nem o benchmark CPU/GPU.
    """
    state = _ACTIVE_TIMING.get()
    if state is None:
        yield
        return
    stack, records = state["stack"], state["records"]
    row = dict(task=state["task"], fold=state["fold"], stage_id=len(records),
               parent_id=stack[-1] if stack else None, depth=len(stack), stage=label,
               started_at=datetime.now().astimezone().isoformat(timespec="seconds"),
               status="running")
    records.append(row)
    stack.append(row["stage_id"])
    prefix = f"{row['task']}: fold {row['fold']:02d} | " + "  " * row["depth"] + label
    print(f"[{format_log_timestamp(row['started_at'])}] {prefix}: início", flush=True)
    start = time.perf_counter()
    try:
        yield
    except BaseException:
        row["status"] = "failed"
        raise
    else:
        row["status"] = "completed"
    finally:
        row["seconds"] = time.perf_counter() - start
        row["finished_at"] = datetime.now().astimezone().isoformat(timespec="seconds")
        children = sum(r["seconds"] for r in records
                       if r["parent_id"] == row["stage_id"] and "seconds" in r)
        row["exclusive_seconds"] = max(0., row["seconds"] - children)
        stack.pop()
        status = "concluído" if row["status"] == "completed" else "interrompido/erro"
        print(f"[{format_log_timestamp(row['finished_at'])}] {prefix}: {status} em "
              f"{format_duration(row['seconds'])}", flush=True)


def timed_function(label):
    """Instrumenta funções sem mudar entradas/saídas; inclui kind/stage no rótulo."""
    def decorate(function):
        """Preserva assinatura/documentação da função e devolve seu invólucro de medição."""
        parameters = signature(function)
        @wraps(function)
        def wrapped(*args, **kwargs):
            """Encaminha argumentos e resultado sem alteração, registrando a duração no contexto ativo."""
            bound = parameters.bind(*args, **kwargs)
            bound.apply_defaults()
            detail = bound.arguments.get("kind", bound.arguments.get("stage"))
            name = f"{label} [{detail}]" if detail is not None else label
            with measure_stage(name):
                return function(*args, **kwargs)
        return wrapped
    return decorate


@contextmanager
def record_fold_timings(task, fold, label):
    """Cria uma coleta independente por fold e restaura o contexto mesmo em erro."""
    state = dict(task=task, fold=int(fold), records=[], stack=[])
    token = _ACTIVE_TIMING.set(state)
    try:
        with measure_stage(label):
            yield state["records"]
    finally:
        _ACTIVE_TIMING.reset(token)


## Admission-level stratification and training-only undersampling


In [ ]:
FIRST_FOLD_DATA_LOAD = None
if os.environ.get('PQFM_SIH_FIRST_FOLD_HILL') == '1':
    dataset_start = time.perf_counter()
    dataset_io_before = Path('/proc/self/io').read_text()
source = DATA_DIR / 'base_modelagem_congelada.parquet'
data = pd.read_parquet(source)
raw_protocol = json.loads((DATA_DIR / 'protocolo.json').read_text())
excluded_features = {'entrada_fim_semana', 'mes_internacao'}
features = [feature for feature in raw_protocol['features']
            if feature not in excluded_features and not feature.startswith('historico_')]
numeric = [c for c in features if pd.api.types.is_numeric_dtype(data[c])]
categorical = [c for c in features if c not in numeric]
for column in categorical:
    data[column] = data[column].astype(object).where(data[column].notna(), None)
protocol = {'features_numericas': numeric, 'features_categoricas': categorical}
GRID_CLASSIFICATION = {'learning_rate':[0.05,0.1,0.2], 'max_depth':[4,6,8], 'min_child_weight':[5,10,20]}
if 'PQFM_XGB_GRID' in os.environ:
    GRID_CLASSIFICATION = json.loads(os.environ['PQFM_XGB_GRID'])
with source.open('rb') as dataset_file:
    dataset_sha256 = hashlib.file_digest(dataset_file, 'sha256').hexdigest()
if os.environ.get('PQFM_SIH_FIRST_FOLD_HILL') == '1':
    FIRST_FOLD_DATA_LOAD = dict(seconds=time.perf_counter()-dataset_start,
                                io_before=dataset_io_before,
                                io_after=Path('/proc/self/io').read_text())

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split

ROUTER_THRESHOLD = 0.60


def balanced_training_rows(indices, y, seed):
    rng = np.random.default_rng(seed)
    classes = [indices[y[indices] == label] for label in (0, 1)]
    size = min(map(len, classes))
    if size == 0:
        raise ValueError('Training needs both classes.')
    return rng.permutation(np.concatenate([rng.choice(rows, size, replace=False) for rows in classes]))


def make_stratified_folds(frame, seed=SEED):
    if not frame.index.equals(pd.RangeIndex(len(frame))) or not frame.id_linha.is_unique:
        raise ValueError('Expected one row per admission and a contiguous index.')
    y = frame.morte.to_numpy()
    if not np.array_equal(np.unique(y), [0, 1]):
        raise ValueError('Expected binary labels.')
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=seed)
    for fold, (training, eval_pool) in enumerate(cv.split(np.zeros(len(y)), y), 1):
        local_seed = seed + fold * 10
        inner, validation = train_test_split(training, test_size=.2, stratify=y[training],
                                              random_state=local_seed)
        calibration, remainder = train_test_split(eval_pool, test_size=.5, stratify=y[eval_pool],
                                                    random_state=local_seed + 1)
        analysis, test = train_test_split(remainder, test_size=.5, stratify=y[remainder],
                                          random_state=local_seed + 2)
        disjoint = np.concatenate([inner, validation, calibration, analysis, test])
        if len(disjoint) != len(frame) or np.unique(disjoint).size != len(frame):
            raise ValueError('Roles overlap or omit admissions.')
        roles = dict(inner_train=balanced_training_rows(inner, y, local_seed + 3),
                     train=balanced_training_rows(training, y, local_seed + 4),
                     validation=validation, calibration=calibration, analysis=analysis, test=test)
        summary = {name: dict(n=len(rows), positives=int(y[rows].sum()), prevalence=float(y[rows].mean()))
                   for name, rows in roles.items()}
        yield dict(fold=fold, repeat=(fold-1)//5+1, **roles, roles=summary)


folds = list(make_stratified_folds(data))
pd.DataFrame([{'fold': fold['fold'], 'repeat': fold['repeat'], **fold['roles']} for fold in folds])


## Classical expert, calibration, and the reference search grid

In [ ]:
def make_preprocessor(protocol):
    """Constrói imputação/escala numérica e one-hot de UF; o chamador deve ajustar apenas no bloco de treino apropriado."""
    return ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median', keep_empty_features=True)), ('scaler', StandardScaler())]), protocol['features_numericas']), ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='__missing__', keep_empty_features=True)), ('one_hot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), protocol['features_categoricas'])], remainder='drop')

def make_estimator(task, params):
    """Cria XGBoost de classificação com os parâmetros aprovados."""
    if task != "classificacao":
        raise ValueError("Este notebook executa somente classificação.")
    return XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", n_estimators=300,
        subsample=0.85, colsample_bytree=0.85, tree_method="hist",
        n_jobs=N_JOBS, random_state=SEED, verbosity=0, **params,
    )

def make_pipeline(task, params, protocol):
    """Encadeia pré-processamento e estimador não ajustados para impedir reutilização de estatísticas entre candidatos."""
    return Pipeline([('preprocessor', make_preprocessor(protocol)), ('model', make_estimator(task, params))])

@timed_function('Clássico: busca interna')
def tune_classic(data, fold, protocol, task, grid):
    """Ajusta candidatos somente em inner_train e seleciona por AP em inner_validation; retorna parâmetros e tabela, sem usar Analysis ou teste externo."""
    features = protocol['features_numericas'] + protocol['features_categoricas']
    target = TARGETS[task]
    train, validation = (data.iloc[fold[role]] for role in ('inner_train', 'inner_validation'))
    for block in (train, validation):
        if block[target].nunique() != 2:
            raise ValueError('Treino e validação de classificação exigem duas classes.')
    preprocessor = make_preprocessor(protocol).fit(train[features])
    train_features = preprocessor.transform(train[features])
    validation_features = preprocessor.transform(validation[features])
    rows = []
    for params in ParameterGrid(grid):
        start = time.perf_counter()
        model = make_estimator(task, params)
        model.fit(train_features, train[target])
        score = average_precision_score(validation[target], model.predict_proba(validation_features)[:, 1])
        if not np.isfinite(score):
            raise ValueError('Métrica interna não finita.')
        rows.append({'params': params, 'score': float(score), 'seconds': time.perf_counter() - start})
    if not rows:
        raise ValueError('Grid vazio.')
    select = max
    best = select(rows, key=lambda row: row['score'])
    return (dict(best['params']), pd.DataFrame(rows))


In [ ]:
def fit_temperature(probability, y):
    """Minimiza NLL nos scores/rótulos da calibração; retorna temperatura positiva, sem alterar pesos do especialista."""
    probability = np.asarray(probability, dtype=float)
    y = np.asarray(y, dtype=int)
    if set(np.unique(y)) != {0, 1}:
        raise ValueError("Calibração exige ambas as classes.")
    logits = logit(np.clip(probability, 1e-7, 1 - 1e-7))

    def nll(log_temperature):
        """Calcula a perda de calibração para log-temperatura; retorna a média da log-loss numericamente estável."""
        scaled = logits / np.exp(log_temperature)
        return float(np.mean(np.logaddexp(0, scaled) - y * scaled))

    lower, upper = np.log(0.01), np.log(100.0)
    result = minimize_scalar(nll, bounds=(lower, upper), method="bounded")
    if not result.success:
        raise RuntimeError("Falha ao ajustar temperatura.")
    candidates = [0.0, lower, upper, float(result.x)]
    return float(np.exp(min(candidates, key=nll)))


def apply_temperature(probability, temperature):
    """Transforma probabilidades usando temperatura já ajustada; não aprende nem consulta rótulos."""
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperatura inválida.")
    return expit(logit(np.clip(probability, 1e-7, 1 - 1e-7)) / temperature)


def best_f1_threshold(y, score):
    """Seleciona o limiar de maior F1 no bloco fornecido de calibração; retorna um score observado, nunca um limiar escolhido no teste."""
    precision, recall, thresholds = precision_recall_curve(y, score)
    if not len(thresholds):
        raise ValueError("Não foi possível determinar um limiar.")
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.argmax(f1))])


def classification_metrics(y, score, prediction):
    """Recebe rótulos, scores e classes previstos; retorna métricas e matriz de confusão, sem ajustar modelos."""
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    both = len(np.unique(y)) == 2
    precision, recall, _ = precision_recall_curve(y, score)
    return {
        "AP": average_precision_score(y, score) if both else np.nan,
        "AUPRC_trapezoidal": auc(recall, precision) if both else np.nan,
        "ROC-AUC": roc_auc_score(y, score) if both else np.nan,
        "Accuracy": accuracy_score(y, prediction),
        "BalancedAccuracy": balanced_accuracy_score(y, prediction),
        "Precision": precision_score(y, prediction, zero_division=0),
        "Recall": recall_score(y, prediction, zero_division=0),
        "F1": f1_score(y, prediction, zero_division=0),
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


In [ ]:
@timed_function('Clássico: ajuste e calibração')
def fit_classic(expert, inner_train, inner_validation, calibration,
                protocol, task, grid):
    # As tabelas recebidas não contêm o teste nem o bloco do roteador.
    """Busca interna e refit em expert; calibration=None desativa ajuste local para usar posteriormente a calibração na validação interna."""
    tuning_data = pd.concat([inner_train, inner_validation], ignore_index=True)
    inner_fold = {
        "inner_train": np.arange(len(inner_train)),
        "inner_validation": np.arange(len(inner_train), len(tuning_data)),
    }
    start = time.perf_counter()
    params, tuning = tune_classic(tuning_data, inner_fold, protocol, task, grid)
    features = protocol["features_numericas"] + protocol["features_categoricas"]
    model = make_pipeline(task, params, protocol)
    with measure_stage('Clássico: reajuste final em expert'):
        model.fit(expert[features], expert[TARGETS[task]])
    fitted = {"pipeline": model, "params": params, "tuning": tuning, "task": task}
    if task == "classificacao" and calibration is not None:
        with measure_stage('Clássico: calibração (temperatura e F1)'):
            probability = model.predict_proba(calibration[features])[:, 1]
            y = calibration[TARGETS[task]]
            temperature = fit_temperature(probability, y)
            score = apply_temperature(probability, temperature)
            fitted.update(temperature=temperature, threshold=best_f1_threshold(y, score))
    fitted["fit_seconds"] = time.perf_counter() - start
    return fitted


In [ ]:
import copy
import torch
import importlib.util
import sys
if importlib.util.find_spec('triton') is not None:
    if 'cudaq' in sys.modules and 'triton' not in sys.modules:
        raise RuntimeError('CUDA-Q foi carregado antes do Triton. Reinicie o kernel e execute as definições em ordem antes do benchmark ou dos experimentos.')
    import triton
from sklearn.decomposition import PCA
torch.set_num_threads(1)

def validate_task_data(X, y, task):
    """Confere matriz X e alvo y finitos, alinhados e compatíveis com a tarefa; retorna arrays validados ou gera erro."""
    X, y = (np.asarray(X, dtype=float), np.asarray(y, dtype=float))
    if task not in TARGETS or X.ndim != 2 or y.ndim != 1 or (len(X) != len(y)) or (not len(y)) or (not np.isfinite(X).all()) or (not np.isfinite(y).all()):
        raise ValueError('Tarefa, features ou alvos inválidos.')
    if task == 'classificacao' and set(np.unique(y)) != {0, 1}:
        raise ValueError('O treinamento do classificador exige ambas as classes.')
    return (X, y)


In [ ]:
def heisenberg_execution_config(device, expectation_method, mps=False, framework="qiskit", precision="double"):
    """Valida framework, expectativas e dispositivo sem fallback entre simuladores."""
    if framework not in ("qiskit", "cudaq", "tilelang"):
        raise ValueError("HEISENBERG_FRAMEWORK deve ser qiskit, cudaq ou tilelang.")
    if precision not in ("single", "double"):
        raise ValueError("HEISENBERG_PRECISION deve ser single ou double.")
    if framework == "qiskit" and mps and precision == "single":
        raise ValueError("Qiskit MPS não suporta precisão single; use double.")
    if device not in ("CPU", "GPU"):
        raise ValueError("HEISENBERG_DEVICE deve ser CPU ou GPU.")
    methods = ("exact",) if framework == "tilelang" else (("statevector", "shots", "exact") if framework == "cudaq" else ("statevector", "shots"))
    if expectation_method not in methods:
        raise ValueError(f"expectation_method para {framework} deve ser um de {methods}.")
    if not isinstance(mps, bool):
        raise ValueError("HEISENBERG_MPS deve ser booleano.")
    if framework == "tilelang" and (mps or precision != "double"):
        raise ValueError("TileLang exige statevector (mps=False) e precisão double.")
    if mps and expectation_method == "statevector":
        raise ValueError("Expectativas exatas com MPS exigem framework='cudaq' e expectation_method='exact'.")
    if framework in ("cudaq", "tilelang") and device != "GPU":
        raise ValueError(f"{framework} requer HEISENBERG_DEVICE='GPU'.")
    if framework == "qiskit" and mps and device != "CPU":
        print("MPS do Qiskit permite apenas CPU; HEISENBERG_DEVICE não se aplica neste modo.", flush=True)
    return dict(
        framework=framework, precision=precision, expectation_method=expectation_method, mps=mps,
        use_gpu_statevector=not mps and device == "GPU",
        statevector_device=None if mps and framework == "qiskit" else device,
    )


In [ ]:
# ÚNICA CÉLULA A EDITAR para configurar o Heisenberg.
HEISENBERG_FRAMEWORK = "tilelang"  # "qiskit", "cudaq" ou "tilelang".
HEISENBERG_PRECISION = "double"  # "single" (FP32) ou "double" (FP64); Qiskit MPS e TileLang exigem double.
HEISENBERG_DEVICE = "GPU"  # "CPU" ou "GPU"; não há fallback automático.
HEISENBERG_EXPECTATION_METHOD = "exact"  # TileLang: "exact"; CUDA-Q: "exact" ou "shots"; Qiskit: "statevector" ou "shots".
HEISENBERG_MPS = False  # True: MPS (Qiskit/CPU ou CUDA-Q/GPU); False: statevector.

HEISENBERG_CONFIG = dict(
    n_components=50, q_enc=26,
    topology="chain", interaction_axes=("x", "y", "z"), encoding_mode="multi_axis", #shared_feature
    hybrid_shap_top_k=50,  # Top features por média de |SHAP| (val-selected Phase3).
    R=1, alpha=0.1,
    use_tanh_scaling=True, measure_2local_diagonal=False, measure_cross_observables=False,  # 3 observáveis locais/qubit.
    shots=4096,  # Usado SOMENTE no modo "shots"; ignorado nas expectativas exatas.
    ideal=True, simulation=True, fakebackend=False,  # Apenas simulação ideal.
    mps_max_bond_dimension=16, mps_truncation_threshold=0.0,  # None: teto automático no CUDA-Q; inteiro: limite explícito. Cutoff zero usa piso numérico no CUDA-Q.
    **heisenberg_execution_config(HEISENBERG_DEVICE, HEISENBERG_EXPECTATION_METHOD, HEISENBERG_MPS, HEISENBERG_FRAMEWORK, HEISENBERG_PRECISION),
)
effective_device = HEISENBERG_CONFIG["statevector_device"] or "CPU"
print("Heisenberg configurado:", HEISENBERG_FRAMEWORK, HEISENBERG_EXPECTATION_METHOD, HEISENBERG_PRECISION,
      f"{'MPS' if HEISENBERG_MPS else 'statevector'}/{effective_device}; {HEISENBERG_CONFIG['q_enc']} qubits")
# O backend efetivo será conferido e exibido ao ajustar cada mapa.


In [ ]:
def make_heisenberg_map(config, seed=SEED):
    """Valida opções de simulação ideal e disponibilidade do dispositivo; retorna mapa público pqfmlib sem fallback nem hardware real."""
    if (not config['ideal'] and not (config.get('topology') == 'native' and config.get('topology_file'))) or not config['simulation'] or config['fakebackend'] or config['measure_2local_diagonal'] or config.get('measure_cross_observables', False):
        raise ValueError('Heisenberg requer simulação ideal e observáveis locais.')
    method = config['expectation_method']
    framework = config.get('framework', 'qiskit')
    device = 'CPU' if config['mps'] and framework == 'qiskit' else config['statevector_device']
    expected = heisenberg_execution_config(device, method, config['mps'], framework, config.get('precision', 'double'))
    if any((config.get(key, 'qiskit' if key == 'framework' else ('double' if key == 'precision' else None)) != value for key, value in expected.items())):
        raise ValueError('Opções contraditórias: use a célula exclusiva de configuração Heisenberg.')
    # if config.get('topology', 'chain') != 'chain':
    #     raise ValueError('Este experimento usa a topologia chain.')
    features_per_edge = len(config['interaction_axes']) if config.get('encoding_mode') == 'multi_axis' else 1
    block_capacity = (config['q_enc'] - 1) * features_per_edge
    if block_capacity < 1 or config['n_components'] < 1 or config['shots'] < 1:
        raise ValueError('O mapa requer capacidade, componentes e shots positivos.')
    if framework == 'qiskit':
        from qiskit_aer import AerSimulator
        if device not in AerSimulator().available_devices():
            raise RuntimeError(f'Aer não disponibiliza {device}. Verifique instalação/driver; sem fallback.')
    from pqfmlib import HeisenbergProjectiveQFM
    return HeisenbergProjectiveQFM(name_file='demanda_pressao_heisenberg', seed=seed, save_circuit_drawings=False, **{key: value for key, value in config.items() if key not in {'n_components', 'hybrid_shap_top_k', 'hybrid_pca_variance'}})

def orient_pca_components(components):
    """Use the first nearly maximal loading to break numerical sign ties."""
    magnitude = np.abs(components)
    anchors = np.argmax(magnitude >= magnitude.max(axis=1, keepdims=True) - 1e-9, axis=1)
    components *= np.sign(components[np.arange(len(components)), anchors])[:, None]


class OneHotCovariancePCA:
    """Covariance PCA with dense numeric and sparse one-hot cross-products."""

    def __init__(self, components, categorical_start):
        self.components = components
        self.categorical_start = categorical_start

    def fit(self, X):
        from scipy.sparse import csr_matrix
        from sklearn.utils.extmath import svd_flip

        n, self.n_features_in_ = X.shape
        numeric = np.ascontiguousarray(X[:, :self.categorical_start])
        categorical = csr_matrix(X[:, self.categorical_start:])
        self.mean_ = np.concatenate([numeric.mean(axis=0), np.asarray(categorical.mean(axis=0)).ravel()])
        cross = (categorical.T @ numeric).T
        gram = np.block([[numeric.T @ numeric, cross],
                         [cross.T, (categorical.T @ categorical).toarray()]])
        gram -= n * np.outer(self.mean_, self.mean_)
        gram /= n - 1
        values, vectors = np.linalg.eigh(gram)
        order = slice(None, -self.components - 1, -1)
        _, components = svd_flip(None, vectors[:, order].T.copy(), u_based_decision=False)
        self.components_ = components
        self.explained_variance_ = values[order].clip(min=0)
        return self

    def transform(self, X):
        return X @ self.components_.T - self.mean_ @ self.components_.T


class HeisenbergFeatures:
    """PCA e mapa congelados; registra operações sem salvar artefatos da biblioteca."""

    def __init__(self, config=None, seed=SEED, factory=make_heisenberg_map):
        """Inicializa HeisenbergFeatures com a configuração recebida; treinamento ocorre somente em fit ou na otimização explícita."""
        self.config = dict(HEISENBERG_CONFIG, **config or {})
        self.seed, self.factory = (seed, factory)
        self.calls = []

    @timed_function('Heisenberg: preparar PCA/mapa')
    def fit(self, X, stage='expert', *, pca_threads=4, structure_only=False, categorical_start=None, stable_pca_sign=False, fitted_pca=None):
        """Ajusta PCA/mapa em X do papel indicado e valida backend; retorna self. Não extrai todas as features: isso cabe a transform."""
        X = np.asarray(X, dtype=float)
        requested = self.config['n_components']
        if X.ndim != 2 or X.shape[0] < 2 or X.shape[1] < 1 or not np.isfinite(X).all():
            raise ValueError('PCA requer matriz finita com pelo menos duas amostras e uma feature.')
        if isinstance(requested, bool) or not isinstance(requested, (int, np.integer)) or requested < 1:
            raise ValueError('n_components deve ser um inteiro positivo.')
        self.n_components_requested_ = int(requested)
        self.n_components_ = min(self.n_components_requested_, X.shape[1], X.shape[0] - 1)
        self.n_zero_slots_ = self.n_components_requested_ - self.n_components_
        print(f'PCA Heisenberg [{stage}]: {self.n_components_}/{requested} componentes; '
              f'{self.n_zero_slots_} posições de encoding preenchidas com zero pelo mapa.', flush=True)
        start = time.perf_counter()
        if fitted_pca is None:
            pca = (PCA(n_components=self.n_components_, svd_solver='covariance_eigh')
                   if categorical_start is None else OneHotCovariancePCA(self.n_components_, categorical_start))
            self.pca_ = pca.fit(X)
            if stable_pca_sign:
                orient_pca_components(self.pca_.components_)
        else:
            if not structure_only or fitted_pca.components_.shape != (self.n_components_, X.shape[1]):
                raise ValueError('Cached PCA is incompatible with this structure-only fit')
            self.pca_ = fitted_pca
        with MAP_FIT_LOCK:
            rng = np.random.get_state()
            try:
                self.map_ = self.factory(self.config, self.seed)
                if structure_only:
                    from pqfmlib import HeisenbergProjectiveQFM
                    if type(self.map_) is not HeisenbergProjectiveQFM:
                        raise TypeError('Structure-only fit requires the reference Heisenberg map.')
                    # This map's fit inspects only input width, then initializes its
                    # angle structure from zeros. PCA itself is fitted to all X.
                    self.map_.fit(np.zeros((1, self.n_components_), dtype=float))
                else:
                    self.map_.fit(self.pca_.transform(X))
            finally:
                np.random.set_state(rng)
        framework = self.config.get('framework', 'qiskit')
        mps = self.config['mps']
        if framework == 'tilelang':
            backend = self.map_.backend
            if backend.precision != self.config['precision'] or backend.device != 'GPU':
                raise RuntimeError('Backend TileLang divergente.')
            self.backend_ = dict(framework=framework, method=backend.target, device=backend.device, precision=backend.precision)
        elif framework == 'cudaq':
            backend = self.map_.backend
            target = 'tensornet-mps' if mps else 'nvidia'
            if backend.precision != self.config.get('precision', 'double') or backend.target != target or backend.mps != mps or backend.mps_max_bond_dimension != self.config['mps_max_bond_dimension'] or (backend.mps_truncation_threshold != self.config['mps_truncation_threshold']):
                raise RuntimeError(f'Backend Heisenberg divergente: target={backend.target}')
            self.backend_ = dict(framework=framework, method=backend.target, device='GPU', precision=backend.precision, bond_dimension=backend.mps_max_bond_dimension, truncation=backend.mps_truncation_threshold)
        else:
            options = self.map_.backend.options
            self.backend_ = {'framework': framework, 'method': str(options.method), 'device': str(options.device), 'precision': str(options.precision), 'bond_dimension': options.matrix_product_state_max_bond_dimension, 'truncation': float(options.matrix_product_state_truncation_threshold)}
            expected = dict(method='matrix_product_state' if mps else 'statevector', device='CPU' if mps else self.config['statevector_device'], precision=self.config.get('precision', 'double'))
            if mps:
                expected.update(
                    bond_dimension=self.config['mps_max_bond_dimension'],
                    truncation=self.config['mps_truncation_threshold'],
                )
            if any((self.backend_[key] != value for key, value in expected.items())):
                raise RuntimeError(f'Backend Heisenberg divergente: {self.backend_}')
        print(f'Heisenberg [{stage}]: {self.backend_['framework']}/{self.backend_['method']}/{self.backend_['device']}; expectativas={self.config['expectation_method']}; {self.config['q_enc']} qubits; mapa preparado, extração ainda não iniciada', flush=True)
        self.n_fit_samples_ = len(X)
        self.calls.append(dict(stage=stage, operation='fit', n_rows=len(X),
                               n_components_requested=self.n_components_requested_,
                               n_components_effective=self.n_components_, zero_encoding_slots=self.n_zero_slots_,
                               seconds=time.perf_counter() - start))
        return self

    @timed_function('Heisenberg: extração')
    def transform(self, X, stage='inference'):
        """Aplica PCA/mapa congelados a X; retorna 3*q_enc features por linha e registra tempo/papel, sem reajuste."""
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.pca_.n_features_in_ or (not np.isfinite(X).all()):
            raise ValueError('Features inválidas para Heisenberg.')
        if not len(X):
            return np.empty((0, 3 * self.config['q_enc']))
        start = time.perf_counter()
        quantum = np.asarray(self.map_.transform(self.pca_.transform(X)), dtype=float)
        if quantum.shape != (len(X), 3 * self.config['q_enc']) or not np.isfinite(quantum).all():
            raise RuntimeError('Saída quântica com dimensão ou valores inválidos.')
        self.calls.append(dict(stage=stage, operation='transform', n_rows=len(X), seconds=time.perf_counter() - start))
        return quantum

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.utils.validation import check_is_fitted
import shap


class HybridSHAPSelector(TransformerMixin, BaseEstimator):
    """Seleciona colunas originais por média de |TreeSHAP|, só no treino recebido."""

    def __init__(self, estimator, top_k=50):
        self.estimator = estimator
        self.top_k = top_k

    def fit(self, X, y, *, training_matrix=None):
        X = np.asarray(X, dtype=float)
        if (isinstance(self.top_k, bool) or not isinstance(self.top_k, (int, np.integer))
                or self.top_k < 1):
            raise ValueError("top_k deve ser um inteiro positivo.")
        if X.ndim != 2 or not len(X) or not X.shape[1] or not np.isfinite(X).all():
            raise ValueError("SHAP requer uma matriz de treino finita e não vazia.")
        self.n_features_in_, self.n_samples_seen_ = X.shape[1], len(X)
        # Mesmo XGBoost/parâmetros do preditor, ajustado antes da seleção.
        if training_matrix is None:
            ranking_model = clone(self.estimator).fit(X, y)
        else:
            ranking_model = _fit_prepared_xgboost(clone(self.estimator), training_matrix)
        explainer = shap.TreeExplainer(ranking_model, feature_perturbation="tree_path_dependent",
                                       model_output="raw")
        total = np.zeros(X.shape[1], dtype=float)
        # Todas as linhas de treino; lotes limitam memória sem amostragem.
        for start in range(0, len(X), 1024):
            batch = X[start:start + 1024]
            values = np.asarray(explainer.shap_values(batch), dtype=float)
            if values.shape != batch.shape or not np.isfinite(values).all():
                raise ValueError("TreeSHAP binário retornou valores inválidos.")
            total += np.abs(values).sum(axis=0)
        self.feature_importances_ = total / len(X)
        # Empates são resolvidos pelo índice original, de forma determinística.
        self.ranking_ = np.argsort(-self.feature_importances_, kind="stable")
        self.n_selected_ = min(self.top_k, X.shape[1])
        self.selected_indices_ = self.ranking_[:self.n_selected_]
        return self

    def transform(self, X):
        check_is_fitted(self, "selected_indices_")
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[1] != self.n_features_in_ or not np.isfinite(X).all():
            raise ValueError("Features incompatíveis com a seleção SHAP ajustada.")
        return X[:, self.selected_indices_]


def quantum_estimator(task, params, config=None):
    """Ajusta ranking SHAP e retreina o XGBoost nas top features, somente no treino."""
    cfg = HEISENBERG_CONFIG if config is None else config
    estimator = make_estimator(task, params)
    return Pipeline([
        ('hybrid_scaler', StandardScaler()),
        ('hybrid_shap', HybridSHAPSelector(estimator, top_k=cfg['hybrid_shap_top_k'])),
        ('model', clone(estimator)),
    ])


def hybrid_selection_diagnostics(expert):
    """Identifica cada coluna concatenada e seu ranking no modelo já ajustado."""
    selector = expert['model'].model.named_steps['hybrid_shap']
    quantum_names = [f'quantum_{axis}_{i}' for i in range(expert['config']['q_enc'])
                     for axis in ('z', 'x', 'y')]
    classic_names = list(expert['preprocessor'].get_feature_names_out())
    names = quantum_names + classic_names
    if len(names) != selector.n_features_in_:
        raise ValueError("Nomes das features divergentes da matriz híbrida.")
    selected = set(selector.selected_indices_.tolist())
    ranking = [dict(rank=rank, index=int(i), feature=names[i],
                    origin='quantum' if i < len(quantum_names) else 'classic',
                    mean_abs_shap=float(selector.feature_importances_[i]), selected=i in selected)
               for rank, i in enumerate(selector.ranking_, start=1)]
    quantum_count = sum(i < len(quantum_names) for i in selected)
    return dict(method='tree_shap_mean_absolute_train', top_k=selector.top_k,
                n_input=selector.n_features_in_, n_selected=selector.n_selected_,
                n_train=selector.n_samples_seen_, n_quantum_selected=quantum_count,
                n_classic_selected=selector.n_selected_ - quantum_count, ranking=ranking)

class HeisenbergExpert:

    def __init__(self, task, feature_map, model):
        """Inicializa HeisenbergExpert com a configuração recebida; treinamento ocorre somente em fit ou na otimização explícita."""
        self.task, self.feature_map, self.model = (task, feature_map, model)

    def predict_raw(self, X, stage='inference'):
        """Extrai features com mapa congelado e aplica estimador treinado; retorna probabilidades ou demanda não negativa."""
        if not len(X):
            return np.empty(0)
        quantum = self.feature_map.transform(X, stage=stage)
        hybrid = np.column_stack([quantum, X])
        return self.model.predict_proba(hybrid)[:, 1]

@timed_function('Heisenberg: seleção interna')
def tune_heisenberg(inner_train, inner_validation, protocol, task, grid, config=None, factory=make_heisenberg_map):
    """Ajusta pré-processamento/PCA/mapa em inner_train, extrai treino/validação uma vez e reutiliza no grid; retorna melhores parâmetros, tabela e chamadas."""
    features = protocol['features_numericas'] + protocol['features_categoricas']
    target = TARGETS[task]
    preprocessing = make_preprocessor(protocol).fit(inner_train[features])
    Xt = preprocessing.transform(inner_train[features])
    Xv = preprocessing.transform(inner_validation[features])
    validate_task_data(Xt, inner_train[target], task)
    if task == 'classificacao' and inner_validation[target].nunique() != 2:
        raise ValueError('Validação de classificação exige ambas as classes.')
    feature_map = HeisenbergFeatures(config, factory=factory).fit(Xt, stage='inner_train')
    qt = np.column_stack([feature_map.transform(Xt, stage='inner_train'), Xt])
    qv = np.column_stack([feature_map.transform(Xv, stage='inner_validation'), Xv])
    rows = []
    with measure_stage('Heisenberg: grid ML sobre features reutilizadas'):
        for params in ParameterGrid(grid):
            start = time.perf_counter()
            model = quantum_estimator(task, params, feature_map.config).fit(qt, inner_train[target])
            score = average_precision_score(inner_validation[target], model.predict_proba(qv)[:, 1])
            if not np.isfinite(score):
                raise ValueError('Métrica interna quântica não finita.')
            rows.append(dict(params=params, score=float(score), seconds=time.perf_counter() - start))
    if not rows:
        raise ValueError('Grid quântico vazio.')
    select = max
    best = dict(select(rows, key=lambda row: row['score'])['params'])
    return (best, pd.DataFrame(rows), feature_map.calls)


## Calibrated experts and the hard router


In [ ]:
ROUTER_DEFAULT_LEARNING_RATE = 0.1

ROUTER_PARAMS = dict(
    objective="binary:logistic", eval_metric="logloss", tree_method="hist",
    n_estimators=100, max_depth=3, subsample=1.0, colsample_bytree=1.0,
    random_state=SEED, n_jobs=N_JOBS, verbosity=0,
)

def router_targets(task, y, classic_prediction, quantum_prediction):
    """Constrói z nas previsões de Analysis: secundário acerta e clássico erra; empates favorecem o clássico."""
    y = np.asarray(y)
    classic_prediction, quantum_prediction = map(np.asarray, (classic_prediction, quantum_prediction))
    if any((v.shape != y.shape or not np.isfinite(v).all() for v in (y, classic_prediction, quantum_prediction))):
        raise ValueError('Vetores inválidos para construir o alvo do roteador.')
    return ((quantum_prediction == y) & (classic_prediction != y)).astype(int)

class ConstantRouter:
    """Decisão determinística quando o bloco de análise tem uma única preferência."""

    def __init__(self, value):
        """Guarda a decisão constante quando Analysis contém uma única classe z; não requer fit."""
        self.value = int(value)

    def predict_proba(self, X):
        """Retorna probabilidades determinísticas 0/1 quando z tem uma classe; mantém interface binária sem inventar exemplos."""
        p = np.full(len(X), self.value, dtype=float)
        return np.column_stack([1 - p, p])

def make_router(learning_rate):
    """Cria XGBoost binário para z; não altera os estimadores dos especialistas."""
    return XGBClassifier(learning_rate=learning_rate, **ROUTER_PARAMS)


In [ ]:
def validation_calibration(raw, y):
    from sklearn.metrics import roc_curve
    temperature = fit_temperature(raw, y)
    probability = apply_temperature(raw, temperature)
    fpr, tpr, thresholds = roc_curve(y, probability)
    usable = np.flatnonzero(np.isfinite(thresholds))
    best = usable[np.argmax((tpr-fpr)[usable])]
    return dict(temperature=temperature, threshold=float(thresholds[best]))


def calibrated_prediction(raw, calibration):
    score = apply_temperature(raw, calibration['temperature'])
    return score, (score >= calibration['threshold']).astype(int)


## Matched 26- and 33-qubit circuit configurations


In [ ]:
SWEEP = {'shap_80': dict(hybrid_shap_top_k=80)}
ENCODINGS = {
    f'q{qubits}_{topology}': dict(q_enc=qubits, n_components=50, R=1, alpha=.1,
                                topology=topology, balanced_encoding=True)
    for qubits in (19, 26, 33) for topology in ('chain', 'plaquettes')
}
display(pd.DataFrame(ENCODINGS).T)


## Shared XGBoost and device helpers


In [ ]:
import xgboost as xgb

def _fit_prepared_xgboost(model, training_matrix):
    # The same quantile cuts and labels are reused by independent histogram fits.
    model._Booster = xgb.train(model.get_xgb_params(), training_matrix,
                               num_boost_round=model.get_params()['n_estimators'])
    model.n_classes_ = 2
    return model


## Precompiled TileLang sweep


In [ ]:
# Opt-in execution benchmark.
import resource
import subprocess
from collections import Counter
from concurrent.futures import Future, ThreadPoolExecutor, as_completed
import itertools
from contextlib import nullcontext
from threading import Condition, Lock, Thread, get_ident
from queue import SimpleQueue


class SweepScheduler:
    """Shared priority queue with bounded admission per visible GPU."""
    def __init__(self, devices, cpu_workers, gpu_workers=1):
        import cupy
        self.cp = cupy
        self.devices = devices
        self.cuda_async_pool = os.environ.get('PQFM_XGB_CUDA_ASYNC_POOL') == '1'
        self.cpu = ThreadPoolExecutor(max_workers=cpu_workers)
        shap_workers = int(os.environ.get('PQFM_SIH_CPU_SHAP_WORKERS', '0'))
        self.cpu_shap = ThreadPoolExecutor(max_workers=shap_workers) if shap_workers else None
        self.jobs = []
        self.ready = Condition()
        self.closing = False
        self.sequence = itertools.count()
        self.threads = [Thread(target=self.worker, args=(device,), daemon=True)
                        for device in devices for _ in range(gpu_workers)]
        for thread in self.threads:
            thread.start()

    def worker(self, device):
        xgb_config = nullcontext()
        if self.cuda_async_pool:
            import xgboost
            xgb_config = xgboost.config_context(use_cuda_async_pool=True)
        with self.cp.cuda.Device(device), xgb_config:
            pool = self.cp.get_default_memory_pool()
            free, total = self.cp.cuda.runtime.memGetInfo()
            pool.set_limit(size=min(2 * 2**30, total // 4))
            while True:
                with self.ready:
                    self.ready.wait_for(lambda: self.closing or self.jobs)
                    if not self.jobs:
                        return
                    index = min(range(len(self.jobs)), key=lambda index: (
                        self.jobs[index][0], device not in self.jobs[index][2], self.jobs[index][1]))
                    _, _, _, future, function, args = self.jobs.pop(index)
                if future.set_running_or_notify_cancel():
                    try:
                        free, _ = self.cp.cuda.runtime.memGetInfo()
                        if free < 384 * 2**20:
                            pool.free_all_blocks()
                            free, _ = self.cp.cuda.runtime.memGetInfo()
                            if free < 384 * 2**20:
                                raise MemoryError(f'GPU {device} has less than 384 MiB available.')
                        result = function(device, *args)
                    except Exception as exc:
                        future.set_exception(exc)
                    else:
                        future.set_result(result)
                del function, args, future

    def gpu(self, priority, function, *args, resident=()):
        future = Future()
        with self.ready:
            self.jobs.append((priority, next(self.sequence), resident, future, function, args))
            self.ready.notify_all()
        return future

    def close(self):
        with self.ready:
            self.closing = True
            self.ready.notify_all()
        for thread in self.threads:
            thread.join()
        self.cpu.shutdown()
        if self.cpu_shap is not None:
            self.cpu_shap.shutdown()


def prepare_fold(fold):
    roles = {name: data.loc[fold[name]] for name in
             ('inner_train', 'validation', 'train', 'calibration', 'analysis', 'test')}
    columns = protocol['features_numericas'] + protocol['features_categoricas']
    inner_pre = make_preprocessor(protocol)
    X = inner_pre.fit_transform(roles['inner_train'][columns])
    V = inner_pre.transform(roles['validation'][columns])
    final_pre = make_preprocessor(protocol)
    fit_raw = final_pre.fit_transform(roles['train'][columns])
    evaluation = {key: final_pre.transform(roles[key][columns])
                  for key in ('calibration', 'analysis', 'test')}
    return roles, X, V, fit_raw, evaluation


def prepare_fold_projections(prepared, width, categorical_start):
    _, inner, validation, outer, evaluation = prepared.result()
    fitted, projected = {}, {}
    for role, training, matrices in (
        ('inner_train', inner, {'inner_train': inner, 'validation': validation}),
        ('expert_refit', outer, {'expert_refit': outer, **evaluation}),
    ):
        components = min(width, training.shape[1], len(training) - 1)
        pca = OneHotCovariancePCA(components, categorical_start).fit(training)
        orient_pca_components(pca.components_)
        fitted[role] = pca
        for name, matrix in matrices.items():
            values = pca.transform(matrix)
            values.setflags(write=False)
            projected[name] = values
    size = sum(values.nbytes for values in projected.values())
    print(f'FOLD_PCA_CACHE width={width} roles={len(projected)} bytes={size}', flush=True)
    return fitted, projected


MAP_FIT_LOCK = Lock()


class FirstFoldRun:
    def __init__(self, settings, scheduler):
        import cupy
        import torch
        import xgboost

        self.cp, self.torch, self.xgb = cupy, torch, xgboost
        self.settings = settings
        self.scheduler = scheduler
        self.artifact = (Path(os.environ['PQFM_SIH_AOT']) / settings.get('artifact', '')
                         if settings.get('aot') else None)
        self.aot_runners = {}
        self.special_runners = {}
        if self.artifact and not all((self.artifact / name).is_file()
                                     for name in ('metadata.json', 'coefficients.so', 'evolve_and_measure.so')):
            raise FileNotFoundError(f'Exported TileLang artifacts are required at {self.artifact}')
        self.grid = list(ParameterGrid(GRID_CLASSIFICATION))
        if not self.grid or not torch.cuda.is_available():
            raise ValueError('The benchmark requires a nonempty grid and a visible CUDA GPU.')
        self.lock = Lock()
        self.events = []
        self.selected = {}
        self.counts = Counter()
        self.threads = settings.get('threads', 4)
        self.config = dict(HEISENBERG_CONFIG, **SWEEP['shap_80']) | settings.get('encoding_config', {})
        self.features = protocol['features_numericas'] + protocol['features_categoricas']

    @contextmanager
    def stage(self, name, device=None, **metadata):
        begin = time.perf_counter()
        cpu = time.thread_time()
        try:
            yield
        finally:
            with self.lock:
                self.events.append(dict(name=name, gpu=device, start=begin-self.start,
                    stop=time.perf_counter()-self.start, calling_thread_cpu=time.thread_time()-cpu, **metadata))

    def device(self, device):
        return nullcontext() if device is None else self.cp.cuda.Device(device)

    def train(self, params, X, y, device, rounds=300, matrix=None, threads=None):
        estimator = make_estimator('classificacao', params).set_params(
            n_estimators=rounds, n_jobs=self.threads if threads is None else threads,
            device='cpu' if device is None else f'cuda:{device}')
        return estimator.fit(X, y) if matrix is None else _fit_prepared_xgboost(estimator, matrix)

    def quantile(self, X, y):
        return self.xgb.QuantileDMatrix(X, label=y, nthread=self.threads)

    def gpu_array(self, X, device):
        return X if device is None else self.cp.asarray(X, dtype=self.cp.float32)

    def host(self, X):
        return self.cp.asnumpy(X) if isinstance(X, self.cp.ndarray) else X

    def shap_scores(self, booster, X, device):
        def reduce_rows(model, rows, gpu):
            model.set_param({'device': 'cpu' if gpu is None else f'cuda:{gpu}'})
            total = np.zeros(rows.shape[1], dtype=np.float64)
            for start in range(0, len(rows), 8192):
                batch = self.gpu_array(rows[start:start+8192], gpu)
                # Booster.predict returns host values. inplace_predict has no TreeSHAP mode.
                matrix = self.xgb.DMatrix(batch, nthread=self.threads)
                values = model.predict(matrix, pred_contribs=True)[:, :-1]
                total += np.abs(values).sum(axis=0, dtype=np.float64)
            return total

        if not isinstance(device, tuple):
            with self.device(device):
                return reduce_rows(booster, X, device) / len(X)

        # Each GPU gets an independent booster and nonoverlapping training rows.
        # Serialize once in memory. XGBoost exposes contributions on the host.
        payload = booster.save_raw()
        boundaries = np.linspace(0, len(X), len(device) + 1, dtype=int)

        def shard(item):
            idx, gpu = item
            with self.device(gpu), self.stage('exact_shap_shard', gpu):
                model = self.xgb.Booster(model_file=payload)
                return reduce_rows(model, X[boundaries[idx]:boundaries[idx+1]], gpu)

        with ThreadPoolExecutor(max_workers=len(device)) as pool:
            return np.sum(list(pool.map(shard, enumerate(device))), axis=0) / len(X)

    def mask(self, model, X, device):
        scores = self.shap_scores(model.get_booster(), X, device)
        return np.argsort(-scores, kind='stable')[:self.config['hybrid_shap_top_k']]

    def search(self, X, V, y, vy, hybrid, gpu_ids):
        stage = 'hybrid' if hybrid else 'classic'
        # Cache quantization by device and ordered column mask. Shared matrices
        # are immutable after construction, and each fit owns its Booster.
        matrices = {}
        arrays = {}
        matrix_lock = Lock()
        preparation_locks = {device: Lock() for device in gpu_ids}

        def prepared(device):
            with preparation_locks.get(device, matrix_lock):
                if device not in arrays:
                    arrays[device] = (self.gpu_array(X, device), self.gpu_array(V, device))
                device_matrices = matrices.setdefault(device, {})
                worker = get_ident()
                if worker not in device_matrices:
                    with self.stage(stage + '_search_prebin', device):
                        device_matrices[worker] = self.quantile(arrays[device][0], y)
                return arrays[device], device_matrices[worker]

        def evaluate(index, rounds, device, phase='all', take=None):
            with self.device(device), self.stage(stage + '_candidate', device, candidate=index, rounds=rounds):
                (Xt, Vt), matrix = prepared(device)
                params = self.grid[index]
                if phase != 'selected':
                    with self.stage(stage + '_candidate_fit', device, candidate=index, rounds=rounds):
                        model = self.train(params, Xt, y, device, rounds, matrix)
                    if phase == 'ranking':
                        return model.get_booster().save_raw()
                if hybrid:
                    if take is None:
                        with self.stage('hybrid_candidate_shap', device, candidate=index, rounds=rounds):
                            take = self.mask(model, Xt, device)
                    with self.stage('hybrid_candidate_selected_bins', device, candidate=index, rounds=rounds):
                        masked_matrix = self.quantile(Xt[:, take], y)
                    with self.stage('hybrid_candidate_selected_fit', device, candidate=index, rounds=rounds):
                        model = self.train(params, None, y, device, rounds, masked_matrix)
                    Vt = Vt[:, take]
                prediction = self.host(model.predict_proba(Vt)[:, 1])
                return index, float(average_precision_score(vy, prediction)), device

        with self.stage(stage + '_search'):
            indexes = sorted(range(len(self.grid)), key=lambda i: -self.grid[i]['max_depth'])
            def wave(candidates, rounds):
                if hybrid and self.scheduler.cpu_shap is not None:
                    ready = SimpleQueue()
                    waiting = {}
                    results = []
                    offload = set(sorted(candidates, key=lambda i: (self.grid[i]['max_depth'], i))[:3])
                    remaining = iter(sorted(candidates, key=lambda i: i not in offload))

                    def watch(job, index, phase):
                        waiting[job] = index, phase
                        job.add_done_callback(ready.put)

                    def submit_ranking(index):
                        phase = 'ranking' if index in offload else 'all'
                        watch(self.scheduler.gpu(10, lambda gpu: evaluate(index, rounds, gpu, phase),
                                                 resident=matrices), index, phase)

                    def cpu_mask(payload, index):
                        with self.stage('hybrid_candidate_shap', None, candidate=index, rounds=rounds):
                            model = self.xgb.Booster(model_file=payload)
                            model.set_param({'nthread': self.threads})
                            scores = self.shap_scores(model, X, None)
                            return np.argsort(-scores, kind='stable')[:self.config['hybrid_shap_top_k']]

                    def submit_selected(index, take):
                        watch(self.scheduler.gpu(8, lambda gpu: evaluate(index, rounds, gpu, 'selected', take),
                                                 resident=matrices), index, 'selected')

                    for _ in range(6):
                        index = next(remaining, None)
                        if index is not None:
                            submit_ranking(index)
                    try:
                        while waiting:
                            job = ready.get()
                            index, phase = waiting.pop(job)
                            value = job.result()
                            if phase == 'ranking':
                                watch(self.scheduler.cpu_shap.submit(cpu_mask, value, index), index, 'shap')
                            elif phase == 'shap':
                                submit_selected(index, value)
                            else:
                                results.append(value)
                                following = next(remaining, None)
                                if following is not None:
                                    submit_ranking(following)
                    finally:
                        for job in waiting:
                            job.cancel()
                    return results
                if not hybrid and self.settings['classic_cpu']:
                    jobs = [self.scheduler.cpu.submit(evaluate, i, rounds, None) for i in candidates]
                else:
                    jobs = [self.scheduler.gpu(10 + offset // len(gpu_ids),
                                               lambda device, i: evaluate(i, rounds, device), i,
                                               resident=matrices)
                            for offset, i in enumerate(candidates)]
                return [job.result() for job in jobs]

            preliminary = []
            if hybrid and (keep := self.settings.get('hybrid_screen_top', 0)):
                preliminary = wave(indexes, 100)
                promoted = sorted(preliminary, key=lambda row: (-row[1], row[0]))[:keep]
                indexes = [row[0] for row in promoted]
            results = wave(indexes, 300)
        winner, score, _ = min(results, key=lambda row: (-row[1], row[0]))
        self.selected[stage] = dict(params=self.grid[winner], validation_ap=score,
                                    round300=sorted(results), checkpoint=sorted(preliminary))
        return self.grid[winner]

    def quantum(self, fmap, X, name, projected=None):
        from pqfmlib.experimental.heisenberg_sih_aot import SIHHeisenbergAOT

        def infer(device, part):
            with self.torch.cuda.device(device):
                key = (fmap, device)
                backend = self.settings.get('quantum_backend')
                if backend in ('commuting_x', 'forest', 'generic_tilelang'):
                    from pqfmlib.experimental.sih_topology_sweep import (
                        CommutingXSweepReadout, ForestSweepReadout, GenericTileLangSweepReadout,
                    )

                    if key not in self.special_runners:
                        if backend != 'generic_tilelang':
                            directory = Path(os.environ['PQFM_SIH_AOT']) / self.settings['artifact']
                            reader = CommutingXSweepReadout if backend == 'commuting_x' else ForestSweepReadout
                            self.special_runners[key] = reader(fmap.map_, directory)
                        else:
                            tuning = self.settings.get('dense_tuning', {})
                            if tuning.get('native_ladder'):
                                from pqfmlib.experimental.sih_native_ladder import NativeLadderReadout

                                self.special_runners[key] = NativeLadderReadout(fmap.map_, tuning['path'])
                            elif tuning.get('frontier'):
                                from pqfmlib.experimental.sih_frontier_tilelang import FrontierSweepReadout

                                self.special_runners[key] = FrontierSweepReadout(fmap.map_, tuning['path'])
                            elif tuning.get('cones'):
                                self.special_runners[key] = SIHHeisenbergAOT(fmap.map_, tuning['path'])
                            else:
                                self.special_runners[key] = GenericTileLangSweepReadout(
                                    fmap.map_, artifacts=tuning.get('path', self.settings.get('dense_artifacts')),
                                    **tuning.get('options', {}))
                    with self.stage(backend, device, rows=len(part), role=name, dense_tuning=self.settings.get('dense_tuning', {}).get('frontier')):
                        return self.special_runners[key].transform(part)
                if key not in self.aot_runners:
                    self.aot_runners[key] = SIHHeisenbergAOT(fmap.map_, self.artifact)
                with self.stage('aot_inference', device, rows=len(part), role=name):
                    return self.aot_runners[key].transform(part)

        with self.stage(name):
            if projected is None:
                with self.stage(name + '_projection', rows=len(X)):
                    projected = fmap.pca_.transform(X)
            pca = projected
            shards = int(os.environ.get('PQFM_SIH_QUANTUM_SHARDS', '3'))
            if shards < 1:
                raise ValueError('PQFM_SIH_QUANTUM_SHARDS must be positive')
            parts = np.array_split(pca, min(shards, len(self.scheduler.devices), len(pca)))
            return np.concatenate([job.result() for job in
                                   [self.scheduler.gpu(1, infer, part) for part in parts]])

    def refit(self, params, X, y, device, phase):
        with self.stage(phase + '_quantile', device):
            matrix = self.xgb.QuantileDMatrix(X, label=y, nthread=8)
        with self.stage(phase + '_boosting', device):
            return self.train(params, None, y, device, matrix=matrix, threads=8)

    def hybrid_fit(self, params, X, y, phase, *, device):
        with self.stage(phase + '_scale', device):
            scaler = StandardScaler().fit(X)
            scaled = self.gpu_array(scaler.transform(X), device)
        with self.stage(phase + '_ranking_fit', device):
            ranking = self.refit(params, scaled, y, device, phase + '_ranking')
        with self.stage(phase + '_exact_shap', device):
            take = self.mask(ranking, scaled, device)
        with self.stage(phase + '_prediction_fit', device):
            model = self.refit(params, scaled[:, take], y, device, phase + '_prediction')
        self.selected[phase + '_mask'] = take.tolist()
        return scaler, take, model, device

    def predict(self, fitted, X, phase):
        scaler, take, model, device = fitted
        with self.device(device), self.stage(phase + '_prediction', device):
            selected = scaler.transform(X)[:, take]
            return self.host(model.predict_proba(self.gpu_array(selected, device))[:, 1])

    def classic_fold(self, prepared):
        roles, X, V, fit_raw, evaluation = prepared.result()
        y = roles['inner_train'].morte.to_numpy()
        vy = roles['validation'].morte.to_numpy()
        hp = self.search(X, V, y, vy, False, self.scheduler.devices)
        labels = roles['train'].morte.to_numpy()

        def refit_and_predict(device):
            with self.stage('classic_expert_refit', device):
                model = self.refit(hp, self.gpu_array(fit_raw, device), labels, device, 'classic_expert')
                return {role: self.host(model.predict_proba(self.gpu_array(raw, device))[:, 1])
                        for role, raw in evaluation.items()}

        scores = (self.scheduler.cpu.submit(refit_and_predict, None) if self.settings['classic_cpu']
                  else self.scheduler.gpu(0, refit_and_predict)).result()
        return scores, self.selected['classic'], dict(seconds=time.perf_counter()-self.start, events=self.events)

    def run(self, fold, prepared, classical, projection_cache=None):
        roles, X, V, fit_raw, evaluation = prepared.result()
        y = roles['inner_train'].morte.to_numpy()
        vy = roles['validation'].morte.to_numpy()
        numeric_width = len(protocol['features_numericas'])

        fitted, projected = projection_cache.result() if projection_cache is not None else ({}, {})
        def fitted_map(matrix, role, stage):
            with self.stage(stage):
                return HeisenbergFeatures(self.config).fit(matrix, stage=role, structure_only=True,
                    categorical_start=numeric_width, stable_pca_sign=True, fitted_pca=fitted.get(role))

        def hybrid_matrix(model, matrix, role):
            return np.column_stack([self.quantum(model, matrix, role, projected.get(role)), matrix])

        def prepare_refit():
            final_map = fitted_map(fit_raw, 'expert_refit', 'expert_refit_pca')
            return (hybrid_matrix(final_map, fit_raw, 'expert_refit'),
                    {role: hybrid_matrix(final_map, matrix, role) for role, matrix in evaluation.items()})

        fmap = fitted_map(X, 'inner_train', 'inner_pca')
        hX = hybrid_matrix(fmap, X, 'inner_train')
        hV = hybrid_matrix(fmap, V, 'validation')
        with self.stage('inner_hybrid_scale'):
            scaler = StandardScaler().fit(hX)
            hX, hV = scaler.transform(hX), scaler.transform(hV)
        refit = (self.scheduler.cpu.submit(prepare_refit)
                 if self.settings.get('prefetch_refit') else None)
        hp = self.search(hX, hV, y, vy, True, self.scheduler.devices)
        del hX, hV
        labels = roles['train'].morte.to_numpy()
        fit_matrix, hybrid_inputs = refit.result() if refit is not None else prepare_refit()
        self.aot_runners.clear()
        self.special_runners.clear()

        def fit_and_score(device):
            model = self.hybrid_fit(hp, fit_matrix, labels, 'outer', device=device)
            return {role: self.predict(model, matrix, role) for role, matrix in hybrid_inputs.items()}

        hybrid = self.scheduler.gpu(0, fit_and_score).result()
        classic, self.selected['classic'], classical_profile = classical.result()
        calibration_y = roles['calibration'].morte.to_numpy()
        ct = validation_calibration(classic['calibration'], calibration_y)
        ht = validation_calibration(hybrid['calibration'], calibration_y)
        analysis = roles['analysis']
        z = router_targets('classificacao', analysis.morte.to_numpy(),
                           calibrated_prediction(classic['analysis'], ct)[1],
                           calibrated_prediction(hybrid['analysis'], ht)[1])

        def route():
            router_pre = make_preprocessor(protocol)
            router_X = router_pre.fit_transform(analysis[self.features])
            gate = (ConstantRouter(z[0]) if np.unique(z).size == 1 else
                    make_router(ROUTER_DEFAULT_LEARNING_RATE).fit(router_X, z))
            return gate.predict_proba(router_pre.transform(roles['test'][self.features]))[:, 1]

        with self.stage('router_fit'):
            w = self.scheduler.cpu.submit(route).result()
        p = calibrated_prediction(classic['test'], ct)[0]
        h = calibrated_prediction(hybrid['test'], ht)[0]
        routed = w > ROUTER_THRESHOLD
        predictions = dict(classic=p, hybrid=h, hard_moe=np.where(routed, h, p), soft_moe=(1-w)*p+w*h)
        y_test = roles['test'].morte.to_numpy()
        curves = {name: precision_recall_curve(y_test, score)[:2] for name, score in predictions.items()}
        result = dict(scores={name: float(average_precision_score(y_test, score))
                            for name, score in predictions.items()},
                    auprc={name: float(auc(recall, precision)) for name, (precision, recall) in curves.items()},
                    routing_fraction=float(routed.mean()), router_positive_fraction=float(z.mean()),
                    shared_classical_profile=classical_profile, selected=self.selected, counts=dict(self.counts), rows=fold['roles'],
                    test_prevalence=float(y_test.mean()), calibration=dict(classic=ct, hybrid=ht),
                    predictions={name: score.tolist() for name, score in predictions.items()})
        if os.environ.get('PQFM_SIH_PAPER_EXTENSION') == '1':
            from pqfmlib.experimental.sih_topology_sweep import routing_diagnostics

            result.update(routing_diagnostics=routing_diagnostics(y_test, p, h, w),
                          router_probabilities=w.tolist())
        return result


def linux_first_fold_snapshot():
    """Process counters and container limits sampled without a profiler dependency."""
    cgroup = Path('/sys/fs/cgroup')

    def fields(path, delimiter=None):
        try:
            records = (line.split(delimiter, 1) for line in Path(path).read_text().splitlines())
            return {k.strip(): int(v.split()[0]) for k, v in records if v.split()[0].isdigit()}
        except (OSError, ValueError, IndexError):
            return {}

    def scalar(name):
        try:
            value = (cgroup / name).read_text().strip()
            return int(value) if value.isdigit() else value
        except OSError:
            return None

    faults = resource.getrusage(resource.RUSAGE_SELF)
    return dict(cpu_seconds=time.process_time(), io=fields('/proc/self/io', ':'),
                status_kib=fields('/proc/self/status', ':'), major_faults=faults.ru_majflt,
                minor_faults=faults.ru_minflt, memory_current=scalar('memory.current'),
                memory_max=scalar('memory.max'), swap_current=scalar('memory.swap.current'),
                memory_stat=fields(cgroup / 'memory.stat'), cpu_stat=fields(cgroup / 'cpu.stat'),
                io_stat=(cgroup / 'io.stat').read_text().strip() if (cgroup / 'io.stat').exists() else None)


def first_fold_trial(name, settings, journal, fold, scheduler, prepared, classical, journal_lock, projection_cache=None):
    run = FirstFoldRun(settings, scheduler)
    before = linux_first_fold_snapshot()
    run.start = time.perf_counter()
    result = run.run(fold, prepared, classical, projection_cache)
    elapsed = time.perf_counter() - run.start
    after = linux_first_fold_snapshot()
    result.update(name=name, settings=settings, seconds=elapsed, events=run.events,
                  fold=int(fold['fold']), grid_parameters=list(run.grid), encoding_config=run.config,
                  repeat=fold['repeat'], dataset_sha256=dataset_sha256, protocol=settings.get('protocol', 'sih_stratified_paper_v1'),
                  fold_fingerprint=sweep_fold_fingerprint(fold),
                  resources=dict(before=before, after=after, scope='process_shared',
                                 mean_process_cores=(after['cpu_seconds']-before['cpu_seconds'])/elapsed))
    with journal_lock, journal.open('a') as output:
        output.write(json.dumps(result) + '\n')
        output.flush()
        os.fsync(output.fileno())
        print(f"STRATIFIED_RESULT {name} {elapsed:.3f} {result['scores']}", flush=True)
    return result


## Compile offline and run the stratified sweep


In [ ]:
def build_stratified_artifacts(root, encodings=None):
    from pqfmlib.experimental.heisenberg_sih_aot import build_sih_kernels, plan_sih_circuit

    for name in ENCODINGS if encodings is None else encodings:
        config = HEISENBERG_CONFIG | SWEEP['shap_80'] | ENCODINGS[name]
        fmap = make_heisenberg_map(config, SEED).fit(np.zeros((1, config['n_components'])))
        shape, *_ = plan_sih_circuit(fmap)
        build_sih_kernels(Path(root) / name, **shape)
        print('STRATIFIED_BUILT', name, shape, flush=True)


def sweep_fold_fingerprint(fold):
    roles = ('inner_train', 'validation', 'train', 'calibration', 'analysis', 'test')
    return hashlib.sha256(json.dumps([np.asarray(fold[role]).tolist() for role in roles]).encode()).hexdigest()


def run_stratified_sweep(artifacts, journal, **kwargs):
    from pqfmlib.experimental.sih_topology_sweep import locked_sweep_journal

    with locked_sweep_journal(Path(journal)) as records:
        return _run_stratified_sweep(artifacts, journal, records=records, **kwargs)


def _run_stratified_sweep(artifacts, journal, *, records, fold_ids=None, encodings=None, profiles=None, trial_folds=None):
    import torch

    os.environ['PQFM_SIH_AOT'] = str(Path(artifacts).resolve())
    journal = Path(journal)
    journal.parent.mkdir(parents=True, exist_ok=True)
    devices = torch.cuda.device_count()
    if devices == 0:
        raise RuntimeError('The stratified sweep requires CUDA.')
    chosen = ENCODINGS.keys() if encodings is None else encodings
    source_folds = folds if trial_folds is None else trial_folds
    selected_folds = [fold for fold in source_folds if fold_ids is None or fold['fold'] in fold_ids]
    if not selected_folds or not chosen:
        raise ValueError('Select at least one valid fold and encoding.')
    limit = int(os.environ.get('PQFM_SIH_IN_FLIGHT', '2'))
    if limit not in (1, 2, 3):
        raise ValueError('Choose 1, 2, or 3 in-flight trials.')
    profile = os.environ.get('PQFM_SIH_CLASSIC_DEVICE', 'gpu')
    if profile not in ('cpu', 'gpu'):
        raise ValueError('PQFM_SIH_CLASSIC_DEVICE must be cpu or gpu.')
    cpu_workers = min(12, max(1, len(os.sched_getaffinity(0)) // 2))
    gpu_workers = int(os.environ.get('PQFM_SIH_GPU_WORKERS', '1'))
    if gpu_workers not in (1, 2, 3):
        raise ValueError('Choose one to three workers per GPU.')
    scheduler = SweepScheduler(tuple(range(devices)), cpu_workers, gpu_workers)
    tag = os.environ.get('PQFM_SIH_TRIAL_TAG', '')
    journal_lock = Lock()
    rows = []
    wall_start = time.perf_counter()
    try:
        with threadpool_limits(limits=2, user_api='blas'), \
                ThreadPoolExecutor(max_workers=limit) as trials, \
                ThreadPoolExecutor(max_workers=1) as classical_pool:
            next_prepared = scheduler.cpu.submit(prepare_fold, selected_folds[0])
            for index, fold in enumerate(selected_folds):
                prepared = next_prepared
                next_prepared = (scheduler.cpu.submit(prepare_fold, selected_folds[index+1])
                                 if index+1 < len(selected_folds) else None)
                fingerprint = sweep_fold_fingerprint(fold)
                pending = []
                for name in chosen:
                    key = f"stratified_{fold['fold']:02d}_{name}{tag}"
                    settings = dict(aot=True, artifact=name, encoding_config=(ENCODINGS[name] if profiles is None else profiles[name]['encoding_config']),
                                    threads=int(os.environ.get('PQFM_XGB_THREADS', '2')), classic_cpu=profile=='cpu',
                                    scheduler='priority_v2', gpu_workers=gpu_workers,
                                    cpu_shap_workers=int(os.environ.get('PQFM_SIH_CPU_SHAP_WORKERS', '0')),
                                    cuda_async_pool=scheduler.cuda_async_pool,
                                    in_flight=limit, prefetch_refit=len(scheduler.devices) > 1 and os.environ.get('PQFM_SIH_PREFETCH_REFIT', '1') == '1', backend_revision=os.environ.get('PQFM_SIH_BACKEND_REVISION', 'unspecified'))
                    if profiles is not None:
                        settings.update(profiles[name])
                    row = records.get(key)
                    if row is None:
                        pending.append((key, settings))
                    elif (row['dataset_sha256'] != dataset_sha256 or row['protocol'] != settings.get('protocol', 'sih_stratified_paper_v1')
                          or row['encoding_config'] != json.loads(json.dumps(HEISENBERG_CONFIG | SWEEP['shap_80'] | settings['encoding_config']))
                          or row['settings']['classic_cpu'] != settings['classic_cpu']
                          or row.get('fold_fingerprint', fingerprint) != fingerprint
                          or row['grid_parameters'] != list(ParameterGrid(GRID_CLASSIFICATION))):
                        raise ValueError(f'Journal mismatch for {key}. Choose a new output path.')
                    else:
                        rows.append(row)
                if pending:
                    use_cache = os.environ.get('PQFM_SIH_RAM_CACHE', '1') == '1'
                    widths = {settings['encoding_config'].get('n_components', HEISENBERG_CONFIG['n_components'])
                              for _, settings in pending}
                    projection_caches = {
                        width: scheduler.cpu.submit(prepare_fold_projections, prepared, width,
                            len(protocol['features_numericas'])) for width in sorted(widths)
                    } if use_cache else {}
                    owner = FirstFoldRun(pending[0][1], scheduler)
                    owner.start = time.perf_counter()
                    classical = classical_pool.submit(owner.classic_fold, prepared)
                    def execute(key, settings):
                        with journal_lock:
                            print(f'STRATIFIED_START {key}', flush=True)
                        width = settings['encoding_config'].get('n_components', HEISENBERG_CONFIG['n_components'])
                        try:
                            return first_fold_trial(key, settings, journal, fold, scheduler, prepared,
                                                    classical, journal_lock, projection_caches.get(width))
                        except Exception:
                            print(f'STRATIFIED_ERROR {key}', flush=True)
                            import traceback
                            traceback.print_exc()
                            raise
                    jobs = [trials.submit(execute, key, settings) for key, settings in pending]
                    rows.extend(job.result() for job in as_completed(jobs))
    finally:
        scheduler.close()
    print('STRATIFIED_SWEEP_SECONDS', round(time.perf_counter()-wall_start, 3), profile, limit, flush=True)
    return pd.DataFrame([dict(name=row['name'], fold=row['fold'], repeat=row['repeat'],
                              encoding=row['settings']['artifact'], seconds=row['seconds'],
                              **row['scores']) for row in rows])


In [ ]:
ARTIFACT_DIR = Path(os.environ.get('PQFM_SIH_STRATIFIED_AOT', 'aot/sih-stratified'))
encoding_setting = os.environ.get('PQFM_SIH_STRATIFIED_ENCODINGS')
chosen_encodings = None if encoding_setting is None else encoding_setting.split(',')
if os.environ.get('PQFM_SIH_BUILD_STRATIFIED') == '1':
    build_stratified_artifacts(ARTIFACT_DIR, chosen_encodings)
if os.environ.get('PQFM_SIH_RUN_STRATIFIED') == '1':
    folds_env = os.environ.get('PQFM_SIH_STRATIFIED_FOLDS')
    result = run_stratified_sweep(
        ARTIFACT_DIR, os.environ.get('PQFM_SIH_STRATIFIED_RESULTS', 'outputs/sih-stratified/results.jsonl'),
        fold_ids=None if folds_env is None else set(map(int, folds_env.split(','))),
        encodings=chosen_encodings)
    display(result)
    display(result.groupby('encoding')[['classic', 'hybrid', 'hard_moe', 'soft_moe']].agg(['mean', 'median']))
    print('STRATIFIED_FINISHED', flush=True)


In [ ]:
# Topology and encoding sweep, with offline native layouts and Qiskit exports.
if os.environ.get('PQFM_SIH_TOPOLOGY_EXPORT') == '1' or os.environ.get('PQFM_SIH_TOPOLOGY_RUN') == '1' or os.environ.get('PQFM_SIH_TOPOLOGY_BUILD') == '1':
    from pqfmlib.experimental.sih_topology_sweep import (
        build_commuting_x, build_forest, commuting_x_plan, extend_topology_manifest,
        forest_plan, prepare_topology_sweep,
    )
    from pqfmlib.experimental.heisenberg_sih_aot import build_sih_kernels, plan_sih_circuit

    complete = os.environ.get('PQFM_SIH_TOPOLOGY_COMPLETE') == '1'
    native_33 = os.environ.get('PQFM_SIH_TOPOLOGY_NATIVE_33') == '1'
    default_root = 'outputs/sih-topologies-complete' if complete else 'outputs/sih-topologies'
    if native_33:
        default_root += '-native33'
    root = Path(os.environ.get('PQFM_SIH_TOPOLOGY_DIR', default_root)).resolve()
    export = os.environ.get('PQFM_SIH_TOPOLOGY_EXPORT') == '1'
    manifest_path = root / 'manifest.json'
    manifest = (prepare_topology_sweep(root, make_heisenberg_map, HEISENBERG_CONFIG, draw=export, complete=complete, native_33=native_33)
                if export or os.environ.get('PQFM_SIH_TOPOLOGY_BUILD') == '1' or not manifest_path.exists() else json.loads(manifest_path.read_text()))
    if os.environ.get('PQFM_SIH_PAPER_EXTENSION') == '1':
        manifest = extend_topology_manifest(manifest)
    if os.environ.get('PQFM_SIH_TOPOLOGY_RUN') == '1' or os.environ.get('PQFM_SIH_TOPOLOGY_BUILD') == '1':
        import fnmatch

        patterns = os.environ.get('PQFM_SIH_TOPOLOGY_MATCH', '*').split(',')
        chosen = [name for name in manifest if any(fnmatch.fnmatchcase(name, pattern) for pattern in patterns)]
        profiles = {}
        artifact_root = root / 'aot'
        row_limit = int(os.environ.get('PQFM_SIH_TOPOLOGY_ROWS', '0'))
        protocol_name = f'sih_topology_stratified_v1_rows{row_limit}'
        screen_top = int(os.environ.get('PQFM_SIH_HYBRID_SCREEN_TOP', '0'))
        if not 0 <= screen_top <= len(list(ParameterGrid(GRID_CLASSIFICATION))):
            raise ValueError('Invalid number of hybrid finalists.')
        if screen_top:
            protocol_name += f'_screen100_top{screen_top}'
        artifact_root /= os.environ.get('PQFM_SIH_BACKEND_REVISION', 'default')
        offline_build = os.environ.get('PQFM_SIH_TOPOLOGY_BUILD') == '1'
        dense_artifacts = os.environ.get('PQFM_SIH_DENSE_AOT')
        tuned_path = os.environ.get('PQFM_SIH_DENSE_TUNING')
        dense_tuning = json.loads(Path(tuned_path).read_text()) if tuned_path else {}
        allow_generic = bool(dense_artifacts) or os.environ.get('PQFM_SIH_TOPOLOGY_GENERIC') == '1'
        if dense_artifacts:
            chosen = [name for name in chosen if manifest[name]['backend'] == 'generic_tilelang']
        blocked = [name for name in chosen if not allow_generic and manifest[name]['backend'] == 'generic_tilelang']
        if blocked:
            print(f'TOPOLOGY_PENDING_BACKEND {len(blocked)} {blocked}', flush=True)
        chosen = [name for name in chosen if allow_generic or manifest[name]['backend'] != 'generic_tilelang']
        for name in chosen:
            entry = manifest[name]
            config = entry['config']
            model = make_heisenberg_map(HEISENBERG_CONFIG | config).fit(np.zeros((1, 50)))
            settings = dict(encoding_config=config, protocol=protocol_name, aot=entry['backend']=='aot', quantum_backend=entry['backend'])
            if screen_top:
                settings['hybrid_screen_top'] = screen_top
            if settings['aot']:
                shape, *_ = plan_sih_circuit(model)
                artifact = '_'.join(str(shape[key]) for key in ('qubits','features','gates','cones','width','depth'))
                path = artifact_root / artifact
                if offline_build and not (path / 'metadata.json').exists():
                    build_sih_kernels(path, **shape)
                    print('TOPOLOGY_BUILT', name, shape, flush=True)
                settings['artifact'] = artifact
            elif entry['backend'] == 'commuting_x':
                degree = commuting_x_plan(model)[1].shape[1]
                artifact = f'commuting_x_{model.q_enc}_50_{degree}'
                path = artifact_root / artifact
                if offline_build and not (path / 'metadata.json').exists():
                    build_commuting_x(path, model.q_enc, 50, degree, batch=1024, threads=128)
                    print('TOPOLOGY_BUILT', name, artifact, flush=True)
                settings['artifact'] = artifact
            elif entry['backend'] == 'forest':
                edges = len(forest_plan(model)[1])
                artifact = f'forest_{model.q_enc}_50_{edges}'
                path = artifact_root / artifact
                if offline_build and not (path / 'metadata.json').exists():
                    build_forest(path, model.q_enc, 50, edges)
                    print('TOPOLOGY_BUILT', name, artifact, flush=True)
                settings['artifact'] = artifact
            if dense_artifacts:
                settings.update(dense_artifacts=str(Path(dense_artifacts).resolve()), artifact='dense_aot')
                tuning = dense_tuning.get(name.rsplit('_', 1)[0])
                if tuning:
                    if tuning.get('native_ladder'):
                        switched = 'dense_native_ladder'
                    else:
                        switched = 'dense_frontier' if tuning.get('frontier') else 'dense_packed_cones' if tuning.get('cones') else 'dense_batched'
                    settings.update(dense_tuning=tuning, artifact=switched)
            profiles[name] = settings

        def topology_screen_fold(fold):
            selected = dict(fold)
            for index, (role, summary) in enumerate(fold['roles'].items()):
                limit = row_limit * (4 if role == 'test' else 1)
                if summary['n'] > limit:
                    rows = fold[role]
                    selected[role], _ = train_test_split(rows, train_size=limit, stratify=data.morte.to_numpy()[rows],
                                                        random_state=SEED + 100*fold['fold'] + index)
            selected['roles'] = {role: dict(n=len(selected[role]), positives=int(data.morte.iloc[selected[role]].sum()),
                                           prevalence=float(data.morte.iloc[selected[role]].mean()))
                                 for role in fold['roles']}
            return selected

        fold_ids = set(map(int, os.environ.get('PQFM_SIH_STRATIFIED_FOLDS', '1').split(',')))
        trial_folds = [topology_screen_fold(fold) for fold in folds if fold['fold'] in fold_ids] if row_limit else None
        if os.environ.get('PQFM_SIH_TOPOLOGY_RUN') == '1':
            journal_prefix = 'dense-' if dense_artifacts else ''
            if screen_top:
                journal_prefix += f'screen100-top{screen_top}-'
            result = run_stratified_sweep(artifact_root, root / f'{journal_prefix}results-tilelang-rows{row_limit}.jsonl',
                                     fold_ids=fold_ids, encodings=chosen, profiles=profiles, trial_folds=trial_folds)
            result.to_csv(root / f'{journal_prefix}scores-tilelang-rows{row_limit}.csv', index=False)
            display(result)
